In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Trading_sentiment_platform").getOrCreate()

In [0]:
# Install NLP + transformer libraries
%pip install transformers torch spacy vaderSentiment hf_transfer


In [0]:
# %restart_python

In [0]:
import transformers
import torch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [0]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = ("ProsusAI/finbert")
tokenizer = AutoTokenizer.from_pretrained(model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_model.eval()

In [0]:
# Define sentiment-scoring function 
def get_sentiment(text):
    if text is None or text.strip() == "":
        return (0.0, "neutral", 0.0, 0.0, 0.0)
    
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    labels = ["positive", "neutral", "negative"]
    
    sentiment_label = labels[int(torch.argmax(probs))]
    
    # Compound score = P - N
    compound = float(probs[0] - probs[2])
    
    return (
        compound,
        sentiment_label,
        float(probs[0]),
        float(probs[1]),
        float(probs[2])
    )


In [0]:
bus_df = spark.read.table("workspace.sec_filings.stg_clean_10Ks")\
                .select("cik","filing_date", "bus_text")

display(bus_df)

In [0]:
# load sentiment data into a dataframe
bus_df = spark.read.table("workspace.sec_filings.stg_clean_10Ks")\
                .select("cik","filing_date", "bus_text")

# transform sentiment_df into pandas for finBERT
bus_pd = bus_df.toPandas()

# Apply sentiment scoring
results = []

for idx, row in bus_pd.iterrows():
    score, label, pos, neu, neg = get_sentiment(row["bus_text"])
    
    results.append({
        "cik": row["cik"],
        "filing_date": row["filing_date"],
        "bus_sent_score": score,
        "bus_sent_label": label,
        "bus_pos_prob": pos,
        "bus_neut_prob": neu,
        "bus_neg_prob": neg,
    })

bus_df = spark.createDataFrame(results)
display(bus_df)

In [0]:
risks_df = spark.read.table("workspace.sec_filings.stg_clean_10Ks")\
                .select("cik","filing_date", "risks_text")

# transform sentiment_df into pandas for finBERT
risks_pd = risks_df.toPandas()

# Apply sentiment scoring
results = []

for idx, row in risks_pd.iterrows():
    score, label, pos, neu, neg = get_sentiment(row["risks_text"])
    
    results.append({
        "cik": row["cik"],
        "filing_date": row["filing_date"],
        "risks_sent_score": score,
        "risks_sent_label": label,
        "risks_pos_prob": pos,
        "risks_neut_prob": neu,
        "risks_neg_prob": neg,
    })

risks_df = spark.createDataFrame(results)
display(risks_df)

In [0]:
mda_df = spark.read.table("workspace.sec_filings.stg_clean_10Ks")\
                .select("cik","filing_date", "mda_text")

# transform mda_df into pandas for finBERT
mda_pd = mda_df.toPandas()

# Apply mda scoring
results = []

for idx, row in mda_pd.iterrows():
    score, label, pos, neu, neg = get_sentiment(row["mda_text"])
    
    results.append({
        "cik": row["cik"],
        "filing_date": row["filing_date"],
        "mda_sent_score": score,
        "mda_sent_label": label,
        "mda_pos_prob": pos,
        "mda_neut_prob": neu,
        "mda_neg_prob": neg,
    })

mda_df = spark.createDataFrame(results)
display(mda_df)

In [0]:
from pyspark.sql import functions as F
# Create a schema for the sentiment dataframe to store in a delta table
spark.sql("""CREATE OR REPLACE TABLE sec_filings.fact_sentiment(
    accession_number STRING,
    cik STRING,
    company_name STRING,
    filing_date DATE,
    form_type STRING,
    negative_prob DOUBLE,
    neutral_prob DOUBLE,
    positive_prob DOUBLE,
    sentiment_label STRING,
    sentiment_score DOUBLE,
    market_cap LONG,
    sector STRING,
    processed_timestamp TIMESTAMP
)
USING DELTA;""")

# Add timestamp to sentiment dataframe
sentiment_df = sentiment_df.withColumn("processed_timestamp", F.current_timestamp())
display(sentiment_df)

In [0]:
# Add sentiment_df data into fact_sentiment delta table
sentiment_df.write.format("delta").mode("append").saveAsTable("sec_filings.fact_sentiment")